 # 04 · Orquestación del Modelo y Catálogos de Gobierno — YelpConnect

 **Tipo:** Orquestación técnica  
 **Dominio:** Gobierno / Data Quality / Trust & Safety  
 **Modo:** Secuencial · Fail-Fast

 ---

 ## Propósito

 Ejecutar de forma controlada los tres notebooks responsables de preparar
 el modelo físico y los catálogos gobernados de YelpConnect:

 ```text
 02_configurar_modelo_gobierno
             ↓
 01_publicar_catalogo_reglas_dq_silver
             ↓
 02_publicar_catalogo_reglas_ts
 ```

 ---

 ## Principio de ejecución

 El proceso utiliza estrategia **Fail-Fast**:

 - si falla la configuración del modelo físico, no se publican catálogos;
 - si falla el catálogo DQ, no se publica el catálogo T&S;
 - solo se considera exitosa la orquestación cuando los tres pasos terminan correctamente.

 Cada notebook se ejecuta mediante `dbutils.notebook.run()`.

 Esto permite mantener cada etapa aislada y conservar claramente sus
 responsabilidades.


In [0]:
# COMMAND ----------

# ============================================================
# 1. CONFIGURACIÓN
# ============================================================

TIMEOUT_SECONDS = 1200

STEPS = [

    {
        "orden": 1,
        "nombre": "Configurar modelo de gobierno",
        "notebook": "./02_configurar_modelo_gobierno"
    },

    {
        "orden": 2,
        "nombre": "Publicar catálogo DQ Bronze",
        "notebook": "../E5_calidad_datos/00_publicar_catalogo_reglas_dq_bronze"
    },

    {
        "orden": 3,
        "nombre": "Publicar catálogo DQ Silver",
        "notebook": "../E5_calidad_datos/01_publicar_catalogo_reglas_dq_silver"
    },

    {
        "orden": 4,
        "nombre": "Publicar catálogo Trust & Safety",
        "notebook": "../E6_controles_gobierno/02_publicar_catalogo_reglas_ts"
    }
]


print("=" * 78)
print("YELPCONNECT - ORQUESTACIÓN DE GOBIERNO")
print("=" * 78)

print(
    f"Pasos configurados : {len(STEPS)}"
)

print(
    f"Timeout por paso   : {TIMEOUT_SECONDS} segundos"
)

print("=" * 78)

# COMMAND ----------


 ## 1 · Secuencia gobernada

 ### Paso 1

 `02_configurar_modelo_gobierno`

 Garantiza que existan:

 - `dq_reglas`
 - `dq_resultados`
 - `evaluation_rule`
 - `evaluation_rule_version`
 - `evaluation_run`

 ### Paso 2

 `01_publicar_catalogo_reglas_dq_silver`

 Publica:

 **22 reglas Data Quality Silver**

 ### Paso 3

 `02_publicar_catalogo_reglas_ts`

 Publica:

 - R001-R006 como reglas activas;
 - R007 como regla experimental `DRAFT`;
 - sus versiones, parámetros, thresholds y pesos.

 ---

 ## Importante

 Este notebook no contiene las reglas.

 **Orquesta.**

 Las definiciones permanecen dentro de sus respectivos notebooks
 productores.



In [0]:
# COMMAND ----------

# ============================================================
# 2. IMPORTS
# ============================================================

from datetime import datetime, timezone

# COMMAND ----------

# ============================================================
# 3. FUNCIÓN DE TIEMPO UTC
# ============================================================

def utc_now():

    return datetime.now(
        timezone.utc
    )




 ## 2 · Evidencia de ejecución

 Para cada etapa se registra:

 - orden;
 - nombre;
 - notebook;
 - estado;
 - inicio UTC;
 - fin UTC;
 - duración;
 - mensaje de error.

 Este registro corresponde a la ejecución del wrapper y no reemplaza
 las tablas operativas `dq_resultados` o `evaluation_run`.

In [0]:
# COMMAND ----------

# ============================================================
# 4. EJECUTAR PIPELINE
# ============================================================

results = []


for step in STEPS:

    orden = step["orden"]
    nombre = step["nombre"]
    notebook_path = step["notebook"]

    started_at = utc_now()

    print()
    print("=" * 78)

    print(
        f"PASO {orden}/{len(STEPS)}"
    )

    print(
        f"Nombre   : {nombre}"
    )

    print(
        f"Notebook : {notebook_path}"
    )

    print(
        f"Inicio   : {started_at.isoformat()}"
    )

    print("=" * 78)

    try:

        output = dbutils.notebook.run(
            notebook_path,
            TIMEOUT_SECONDS,
            {}
        )

        finished_at = utc_now()

        duration_seconds = (
            finished_at
            -
            started_at
        ).total_seconds()

        results.append(
            {
                "orden": orden,
                "nombre": nombre,
                "notebook": notebook_path,
                "estado": "SUCCESS",
                "inicio_utc": started_at.isoformat(),
                "fin_utc": finished_at.isoformat(),
                "duracion_segundos": round(
                    duration_seconds,
                    2
                ),
                "error": ""
            }
        )

        print()
        print(
            f"✔ PASO {orden} COMPLETADO"
        )

        print(
            f"Duración: "
            f"{duration_seconds:.2f} segundos"
        )

        if output:

            print(
                f"Salida: {output}"
            )

    except Exception as exc:

        finished_at = utc_now()

        duration_seconds = (
            finished_at
            -
            started_at
        ).total_seconds()

        error_message = str(exc)

        results.append(
            {
                "orden": orden,
                "nombre": nombre,
                "notebook": notebook_path,
                "estado": "FAILED",
                "inicio_utc": started_at.isoformat(),
                "fin_utc": finished_at.isoformat(),
                "duracion_segundos": round(
                    duration_seconds,
                    2
                ),
                "error": error_message[:4000]
            }
        )

        print()
        print(
            f"❌ PASO {orden} FALLÓ"
        )

        print(
            f"Notebook: {notebook_path}"
        )

        print(
            f"Error: {error_message}"
        )

        # --------------------------------------------
        # Mostrar evidencia antes de detener ejecución
        # --------------------------------------------

        df_resultados = (
            spark.createDataFrame(
                results
            )
        )

        display(
            df_resultados
            .orderBy(
                "orden"
            )
        )

        raise RuntimeError(
            f"""
            ORQUESTACIÓN DETENIDA

            Paso:
            {orden}

            Notebook:
            {notebook_path}

            Causa:
            {error_message}
            """
        ) from exc

# COMMAND ----------


 ## 3 · Resultado de la orquestación

 Si la ejecución llega a esta sección significa que los tres notebooks
 finalizaron correctamente.

 La tabla mostrada a continuación permite verificar la duración y estado
 individual de cada etapa.


In [0]:
# COMMAND ----------

# ============================================================
# 5. RESULTADOS
# ============================================================

df_resultados = (
    spark.createDataFrame(
        results
    )
)


display(
    df_resultados
    .orderBy(
        "orden"
    )
)


In [0]:
 # COMMAND ----------

# ============================================================
# 6. VALIDACIÓN GLOBAL
# ============================================================

failed_steps = (
    df_resultados

    .filter(
        "estado <> 'SUCCESS'"
    )

    .count()
)


if failed_steps > 0:

    raise Exception(
        f"""
        ❌ Existen {failed_steps}
        pasos fallidos.
        """
    )


successful_steps = (
    df_resultados

    .filter(
        "estado = 'SUCCESS'"
    )

    .count()
)


if successful_steps != len(
    STEPS
):

    raise Exception(
        f"""
        ❌ Ejecución incompleta.

        Esperados:
        {len(STEPS)}

        Exitosos:
        {successful_steps}
        """
    )


print(
    "✔ Todos los pasos finalizaron correctamente."
)

# COMMAND ----------

 ## Definition of Done

 La orquestación se considera exitosa cuando:

 - [ ] `02_configurar_modelo_gobierno` finaliza correctamente;
 - [ ] `01_publicar_catalogo_reglas_dq_silver` finaliza correctamente;
 - [ ] `02_publicar_catalogo_reglas_ts` finaliza correctamente;
 - [ ] existen 3 ejecuciones con estado `SUCCESS`;
 - [ ] ningún paso fue omitido;
 - [ ] una falla intermedia detiene las etapas posteriores.

 ---

 ## Resultado funcional

 Después de una ejecución exitosa quedan disponibles:

 ```text
 workspace.yelp_gov
 │
 ├── dq_reglas
 │      └── 22 reglas Silver
 │
 ├── dq_resultados
 │
 ├── evaluation_rule
 │      └── R001-R007
 │
 ├── evaluation_rule_version
 │      ├── R001-R006 ACTIVE
 │      └── R007 DRAFT
 │
 └── evaluation_run
 ```

 ---

 ## Siguiente etapa

 El modelo y los catálogos de gobierno quedan preparados.

 La siguiente etapa del procesamiento es:

 `E7_pipeline_lakehouse/01_ingestar_bronze`


In [0]:
# COMMAND ----------

# ============================================================
# 7. RESUMEN FINAL
# ============================================================

total_duration = sum(
    row["duracion_segundos"]
    for row in results
)


print()
print("=" * 78)
print("ORQUESTACIÓN DE GOBIERNO COMPLETADA")
print("=" * 78)

print()
print(
    f"Pasos ejecutados : "
    f"{successful_steps}/{len(STEPS)}"
)

print(
    f"Duración total   : "
    f"{total_duration:.2f} segundos"
)

print()

for result in results:

    print(
        f"✔ {result['orden']} · "
        f"{result['nombre']} "
        f"({result['duracion_segundos']} s)"
    )

print()
print(
    "Modelo físico    : ✔"
)

print(
    "Catálogo DQ      : ✔"
)

print(
    "Catálogo T&S     : ✔"
)

print()
print(
    "Siguiente paso:"
)

print(
    "→ E7_pipeline_lakehouse/01_ingestar_bronze"
)

print("=" * 78)